In [1]:
%pip install ib_insync

   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------- ----------------------------- 3.1/12.3 MB 15.0 MB/s eta 0:00:01
   ---------------- ----------------------- 5.0/12.3 MB 13.0 MB/s eta 0:00:01
   ---------------------- ----------------- 7.1/12.3 MB 11.2 MB/s eta 0:00:01
   ---------------------------- ----------- 8.7/12.3 MB 10.3 MB/s eta 0:00:01
   ---------------------------------- ----- 10.5/12.3 MB 10.2 MB/s eta 0:00:01
   ---------------------------------------  12.1/12.3 MB 10.0 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 9.5 MB/s  0:00:01

   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from ib_insync import IB

ib = IB()
ib.connect('127.0.0.1', 4002, clientId=3)

print("Connecté :", ib.isConnected())
print("Account ID :", ib.managedAccounts())

#ib.disconnect()

In [2]:
import math
import numpy as np

In [ ]:
def norm_cdf(x):
    """
    cumulative distribution function (intégrale de la PDF)
    CDF de la loi normale standard N(x).
    Implémentée from scratch via math.erf (stdlib Python).
    N(x) = 0.5 * (1 + erf(x / sqrt(2)))
    """
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

In [ ]:
def norm_pdf(x):
    """
    Probability Density Function (gaussienne)
    PDF de la loi normale standard n(x).
    n(x) = exp(-x²/2) / sqrt(2π)
    """
    return math.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)

In [ ]:
def forward(S, r, q, T):
    """
    Forward carry-based (Eq. 3 du roadmap).
    F(T) = S * exp((r - q) * T)
 
    S : spot de référence (mid bid/ask)
    r : taux sans risque continu
    q : dividend yield ou carry implicite
    T : maturité en fractions d'année
    """
    return S * math.exp((r - q) * T)

In [ ]:
def log_moneyness(K, F):
    """
    Log-moneyness relatif au forward (Eq. 6).
    k = ln(K / F)
 
    k < 0 : OTM put / ITM call
    k = 0 : ATM forward
    k > 0 : OTM call / ITM put
    """
    return math.log(K / F)

In [ ]:
def total_variance(sigma, T):
    """
    Variance totale (Eq. 7).
    w = sigma² * T
    """
    return sigma ** 2 * T

In [ ]:
def compute_d1(S, K, T, r, q, sigma):
    """
    d1 (Eq. 8) : [ln(S/K) + (r - q + σ²/2) * T] / (σ * √T)
    d2 (Eq. 9) : d1 - σ * √T
    """
    sqrt_T = math.sqrt(T)
    d1 = (math.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * sqrt_T)
    return d1

In [ ]:
def compute_d2(d1, sigma):
    """
    d1 (Eq. 8) : [ln(S/K) + (r - q + σ²/2) * T] / (σ * √T)
    d2 (Eq. 9) : d1 - σ * √T
    """
    sqrt_T = math.sqrt(T)
    d2 = d1 - sigma * sqrt_T
    return d2

In [ ]:
def bs_call(S, K, T, r, q, sigma):
    """
    Prix call européen Black-Scholes (Eq. 10).
    C = S*e^(-qT)*N(d1) - K*e^(-rT)*N(d2)
    """
    d1 = compute_d1(S, K, T, r, q, sigma)
    d2 = compute_d2(d1, sigma)
    return S * math.exp(-q * T) * norm_cdf(d1) - K * math.exp(-r * T) * norm_cdf(d2)

In [ ]:
def bs_put(S, K, T, r, q, sigma):
    """
    Prix put européen Black-Scholes (Eq. 11).
    P = K*e^(-rT)*N(-d2) - S*e^(-qT)*N(-d1)
    """
    d1 = compute_d1(S, K, T, r, q, sigma)
    d2 = compute_d2(d1, sigma)
    return K * math.exp(-r * T) * norm_cdf(-d2) - S * math.exp(-q * T) * norm_cdf(-d1)

In [ ]:
def bs_price(S, K, T, r, q, sigma, right):
    """
    Prix Black-Scholes pour call ('C') ou put ('P').
    """
    if right == 'C':
        return bs_call(S, K, T, r, q, sigma)
    elif right == 'P':
        return bs_put(S, K, T, r, q, sigma)
    else:
        raise ValueError(f"right doit être 'C' ou 'P', reçu : {right}")